# benchmark2: logGP vs linGP（RT-1 / 200x200 camera benchmark）

`logGP_compare2.ipynb` の目的（RT-1トモグラフィにおける `linGP` / `logGP` 比較）を、
`gpoloidal + zray` を直接使う形で簡潔に整理した notebook です。

この notebook では以下を行います。
- RT-1 幾何に対して forward operator `H` を生成（`zray` + `gpoloidal`）
- 同じ prior スケールで `linGP` と `logGP` を推定
- SNR を振って誤差指標を比較（平均・分散）


In [ ]:
%matplotlib inline
# NOTE: autoreload is intentionally disabled on Windows (cp932) due an IPython/deduperreload Unicode bug.

import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import zray
import gpoloidal.rt1 as rt1
import gpoloidal.tomography as tomo

warnings.filterwarnings("ignore", category=RuntimeWarning, module=r"gpoloidal\.rt1\.mag")
np.set_printoptions(precision=4, suppress=True)


## 実験設定

計算時間を抑えるために以下を軽量化しています。
- カメラ解像度を縮小（例: `48x48`）
- 誘導点を間引き（`inducing_stride`）
- 反射なし（`nreflections=0`）


In [ ]:
@dataclass
class ExperimentConfig:
    point_file: str = "example/rt1tomography/point_temp.npz"
    inducing_stride: int = 6
    resolution: tuple[int, int] = (200, 200)
    Lnum: int = 41
    nreflections: int = 0
    phantom_name: str = "Hollow"
    length_scale_factor: float = 1.4
    snr_rms_targets: tuple[float, ...] = (100, 30, 10, 3, 1)
    n_trials: int = 2
    seed: int = 42
    log_prior_mean: float = -3.0
    log_bound_value: float = -5.0
    bound_sig: float = 0.1
    max_log_iters: int = 20
    log_tol: float = 1e-5

cfg = ExperimentConfig()
cfg


## ヘルパー関数

- `prepare_rt1_problem`: kernel / ray / forward matrix を構築
- `fit_lingp`: 線形ガウスモデルの閉形式 posterior
- `fit_loggp`: `GPT_log_general` による Laplace 近似


In [ ]:
def prepare_rt1_kernel(point_file: str, inducing_stride: int):
    """Load RT-1 kernel and optionally subsample inducing points for speed."""
    k = rt1.Kernel2D_scatter_rt1()
    pt = np.load(point_file)

    idx = np.arange(pt["r_idc"].size)
    if inducing_stride > 1:
        idx = idx[::inducing_stride]

    k.load_point(
        r_idc=pt["r_idc"][idx],
        z_idc=pt["z_idc"][idx],
        r_bd=pt["r_bd"],
        z_bd=pt["z_bd"],
        length_sq_fuction=rt1.phantom.Length_scale_sq,
        is_plot=False,
    )
    return k


def build_ray_model(vessel, resolution=(48, 48)):
    camera = zray.measurement.Camera2D_rphiz(
        focal_length=0.01,
        location=(1.2, 0.0, 0.0),
        center_angles=(23, 0),
        sensor_size=(0.0082, 0.0082),
        resolution=resolution,
        rotation=0.0,
    )
    return zray.Raytracing(vessel, camera)


def prepare_rt1_problem(cfg: ExperimentConfig):
    k = prepare_rt1_kernel(cfg.point_file, cfg.inducing_stride)

    # Grid interface is used only for visualization / interpolation.
    r_plot = np.linspace(0.05, 1.05, 501)
    z_plot = np.linspace(-0.7, 0.7, 501)
    k.set_grid_interface(r_plot=r_plot, z_plot=z_plot, add_bound=True)

    raymodel = build_ray_model(k.vessel, cfg.resolution)
    raymodel.main(nreflections=cfg.nreflections, pass_through_first=True)
    ray = raymodel.rays[0]

    H = k.create_obs_matrix_kernel_weighting(ray=ray, Lnum=cfg.Lnum)

    phantom = rt1.phantom.get_phantom_funtion(cfg.phantom_name)
    f_true = np.asarray(phantom(k.r_idc, k.z_idc), dtype=float)
    f_true = np.clip(f_true, 0.0, None)
    g_true = H @ f_true

    return {
        "kernel": k,
        "raymodel": raymodel,
        "ray": ray,
        "H": H,
        "f_true": f_true,
        "g_true": g_true,
        "r_plot": r_plot,
        "z_plot": z_plot,
    }


def fit_lingp(H, g_obs, obs_noise_std, Kf_pri, muf_pri, eps=1e-6):
    """Closed-form posterior for linear Gaussian model y = H f + e."""
    H = np.asarray(H, dtype=float)
    g_obs = np.asarray(g_obs, dtype=float)
    obs_noise_std = np.asarray(obs_noise_std, dtype=float)
    K = 0.5 * (Kf_pri + Kf_pri.T) + eps * np.eye(Kf_pri.shape[0])
    m0 = np.asarray(muf_pri, dtype=float)

    K_inv = np.linalg.inv(K)
    w = 1.0 / obs_noise_std
    sigiH = w[:, None] * H
    A = sigiH.T @ sigiH + K_inv
    rhs = sigiH.T @ (w * g_obs) + K_inv @ m0

    m = np.linalg.solve(A, rhs)
    K_pos = np.linalg.inv(A)
    return m, K_pos


def fit_loggp(H, g_obs, obs_noise_std, Kf_pri, muf_pri, *, max_iters=30, tol=1e-5, init=None):
    """Laplace approximation for positive latent field via log-GP."""
    obs_noise_std = np.asarray(obs_noise_std, dtype=float)
    model = tomo.GPT_log_general(H=H, Kf_pri=Kf_pri, muf_pri=muf_pri)
    model.set_obs(g_obs=np.asarray(g_obs, dtype=float), obs_noise_profile=np.ones_like(obs_noise_std), normalize=False, obs_noise_level=float(np.mean(obs_noise_std)))

    if init is None:
        f = np.asarray(muf_pri, dtype=float).copy()
    else:
        f = np.asarray(init, dtype=float).copy()

    loss_history = []
    for _ in range(max_iters):
        delta_f, loss = model.update(f)
        f = np.clip(f + delta_f, -12.0, 8.0)
        loss_history.append(float(loss))
        if loss < tol:
            break

    model.postprocess(f)
    return {
        "model": model,
        "f_latent": f,
        "f_mean": model.expf_mean,
        "f_median": model.expf_median,
        "f_std": model.expf_std,
        "loss_history": loss_history,
    }


def field_metrics(f_est, f_true, *, name: str):
    f_est = np.asarray(f_est, dtype=float)
    f_true = np.asarray(f_true, dtype=float)
    diff = f_est - f_true
    rmse = float(np.sqrt(np.mean(diff**2)))
    mae = float(np.mean(np.abs(diff)))
    rel_rmse = float(np.linalg.norm(diff) / (np.linalg.norm(f_true) + 1e-12))
    corr = float(np.corrcoef(f_est, f_true)[0, 1])
    neg_frac = float(np.mean(f_est < 0))
    return {
        f"{name}_rmse": rmse,
        f"{name}_mae": mae,
        f"{name}_rel_rmse": rel_rmse,
        f"{name}_corr": corr,
        f"{name}_neg_frac": neg_frac,
    }



## Forward model と真値の準備


In [ ]:
problem = prepare_rt1_problem(cfg)

k = problem["kernel"]
H = problem["H"]
f_true = problem["f_true"]
g_true = problem["g_true"]
raymodel = problem["raymodel"]

print(f"nI={k.nI}, nb={k.nb}, H.shape={H.shape}, resolution={cfg.resolution}")
print(f"g_true mean={g_true.mean():.4e}, min={g_true.min():.4e}, max={g_true.max():.4e}")


In [ ]:
K_lin_pri, mu_lin_pri = k.set_kernel(
    length_scale_factor=cfg.length_scale_factor,
    is_bound=True,
    bound_value=0.0,
    bound_sig=cfg.bound_sig,
    mean=0.0,
)

K_log_pri, mu_log_pri = k.set_kernel(
    length_scale_factor=cfg.length_scale_factor,
    is_bound=True,
    bound_value=cfg.log_bound_value,
    bound_sig=cfg.bound_sig,
    mean=cfg.log_prior_mean,
)

print(K_lin_pri.shape, K_log_pri.shape)


In [ ]:
f_true_hd = k.convert_grid(f_true, boundary=0.0)
mask_hd = k.mask
im_kwargs_hd = k.im_kwargs

fig, axs = plt.subplots(1, 2, figsize=(9, 3.8), constrained_layout=True)
axs[0].imshow(f_true_hd * mask_hd, **im_kwargs_hd, cmap="turbo")
axs[0].set_title("Ground Truth (inducing->grid)")
k.plt_rt1_flux(ax=axs[0], linewidths=0.7)

axs[1].imshow(g_true.reshape(cfg.resolution), origin="lower", cmap="magma")
axs[1].set_title("Forward image g_true")
for ax in axs:
    ax.set_xlabel("R or pixel-x")
    ax.set_ylabel("Z or pixel-y")
plt.show()


## ノイズ依存性ベンチマーク（linGP vs logGP）

各 SNR について複数 trial のノイズを生成し、以下を記録します。
- `RMSE`, `relative RMSE`, `correlation`
- `linGP` の負値率（`logGP` は原理的に正）
- `logGP` の収束反復数


In [ ]:
def run_noise_sweep(
    cfg: ExperimentConfig,
    H,
    f_true,
    g_true,
    K_lin_pri,
    mu_lin_pri,
    K_log_pri,
    mu_log_pri,
):
    rng_master = np.random.default_rng(cfg.seed)
    records = []
    examples = {}

    g_mean = float(np.mean(g_true))
    g_rms = float(np.sqrt(np.mean(np.square(g_true))))

    for target_snr_rms in cfg.snr_rms_targets:
        # Homoscedastic noise: scalar level + flat profile
        obs_noise_level = float(g_rms / target_snr_rms)
        obs_noise_std = np.full_like(g_true, obs_noise_level, dtype=float)

        for trial in range(cfg.n_trials):
            seed_i = int(rng_master.integers(0, 2**31 - 1))
            rng = np.random.default_rng(seed_i)
            g_obs = g_true + obs_noise_std * rng.standard_normal(g_true.size)

            f_lin, _ = fit_lingp(H, g_obs, obs_noise_std, K_lin_pri, mu_lin_pri)
            log_fit = fit_loggp(
                H, g_obs, obs_noise_std,
                K_log_pri, mu_log_pri,
                max_iters=cfg.max_log_iters,
                tol=cfg.log_tol,
            )
            f_log = log_fit["f_mean"]

            rec = {
                "snr_rms_target": float(target_snr_rms),
                "trial": trial,
                "seed": seed_i,
                "obs_noise_level": obs_noise_level,
                "noise_to_mean_ratio": float(obs_noise_level / (g_mean + 1e-12)),
                "noise_to_rms_ratio": float(obs_noise_level / (g_rms + 1e-12)),
                "snr_mean": float(g_mean / (obs_noise_level + 1e-12)),
                "snr_rms": float(g_rms / (obs_noise_level + 1e-12)),
                "log_iters": len(log_fit["loss_history"]),
                "log_last_loss": float(log_fit["loss_history"][-1]) if log_fit["loss_history"] else np.nan,
            }
            rec |= field_metrics(f_lin, f_true, name="lin")
            rec |= field_metrics(f_log, f_true, name="log")

            # Observation-space fit metric (noise-normalized residual)
            g_lin = H @ f_lin
            g_log = H @ f_log
            rec["lin_chi2"] = float(np.mean(((g_lin - g_obs) / (obs_noise_std + 1e-12)) ** 2))
            rec["log_chi2"] = float(np.mean(((g_log - g_obs) / (obs_noise_std + 1e-12)) ** 2))

            records.append(rec)

            if trial == 0:
                examples[target_snr_rms] = {
                    "g_obs": g_obs,
                    "f_lin": f_lin,
                    "f_log": f_log,
                    "log_fit": log_fit,
                    "obs_noise_level": obs_noise_level,
                }

    df = pd.DataFrame.from_records(records)
    df = df.sort_values(["snr_rms_target", "trial"], ascending=[False, True]).reset_index(drop=True)
    return df, examples

results_df, examples = run_noise_sweep(
    cfg=cfg,
    H=H,
    f_true=f_true,
    g_true=g_true,
    K_lin_pri=K_lin_pri,
    mu_lin_pri=mu_lin_pri,
    K_log_pri=K_log_pri,
    mu_log_pri=mu_log_pri,
)

results_df.head()


In [ ]:
summary = (
    results_df.groupby("snr_rms_target")
    .agg(
        n_trials=("trial", "count"),
        lin_rmse_mean=("lin_rmse", "mean"),
        lin_rmse_std=("lin_rmse", "std"),
        log_rmse_mean=("log_rmse", "mean"),
        log_rmse_std=("log_rmse", "std"),
        lin_rel_rmse_mean=("lin_rel_rmse", "mean"),
        log_rel_rmse_mean=("log_rel_rmse", "mean"),
        lin_corr_mean=("lin_corr", "mean"),
        log_corr_mean=("log_corr", "mean"),
        lin_neg_frac_mean=("lin_neg_frac", "mean"),
        log_iters_mean=("log_iters", "mean"),
    )
    .sort_index(ascending=False)
)
summary


In [ ]:
snr_rms_axis = summary.index.to_numpy(dtype=float)

fig, axs = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True)

# RMSE (with std error bars)
axs[0].errorbar(snr_rms_axis, summary["lin_rmse_mean"], yerr=summary["lin_rmse_std"], marker="o", label="linGP")
axs[0].errorbar(snr_rms_axis, summary["log_rmse_mean"], yerr=summary["log_rmse_std"], marker="o", label="logGP")
axs[0].set_xscale("log")
axs[0].invert_xaxis()
axs[0].set_title("Field RMSE vs RMS-based SNR Target")
axs[0].set_xlabel("RMS-based SNR target")
axs[0].set_ylabel("RMSE")
axs[0].grid(alpha=0.3)
axs[0].legend()

# Relative RMSE
axs[1].plot(snr_rms_axis, summary["lin_rel_rmse_mean"], marker="o", label="linGP")
axs[1].plot(snr_rms_axis, summary["log_rel_rmse_mean"], marker="o", label="logGP")
axs[1].set_xscale("log")
axs[1].invert_xaxis()
axs[1].set_title("Field Relative RMSE")
axs[1].set_xlabel("RMS-based SNR target")
axs[1].set_ylabel("||f_hat-f|| / ||f||")
axs[1].grid(alpha=0.3)

# Linear negativity (diagnostic) + log iterations on twin axis
ax2 = axs[2]
ax2.plot(snr_rms_axis, summary["lin_neg_frac_mean"], marker="o", color="tab:red", label="linGP negative fraction")
ax2.set_xscale("log")
ax2.invert_xaxis()
ax2.set_xlabel("RMS-based SNR target")
ax2.set_ylabel("linGP negative fraction", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")
ax2.grid(alpha=0.3)
ax2.set_title("Positivity / logGP convergence")

ax2b = ax2.twinx()
ax2b.plot(snr_rms_axis, summary["log_iters_mean"], marker="s", color="tab:blue", label="logGP iters")
ax2b.set_ylabel("logGP iterations", color="tab:blue")
ax2b.tick_params(axis="y", labelcolor="tab:blue")

plt.show()


## 代表例の可視化（1 trial）

`SNR=3` を優先し、存在しなければ最小 SNR を表示します。


In [ ]:
snr_rms_example = 3.0 if 3.0 in examples else float(sorted(examples.keys())[0])
ex = examples[snr_rms_example]

f_lin_hd = k.convert_grid(ex["f_lin"], boundary=0.0)
f_log_hd = k.convert_grid(ex["f_log"], boundary=0.0)
err_lin_hd = k.convert_grid(ex["f_lin"] - f_true, boundary=0.0)
err_log_hd = k.convert_grid(ex["f_log"] - f_true, boundary=0.0)
mask_hd = k.mask
im_kwargs = k.im_kwargs

fig, axs = plt.subplots(2, 3, figsize=(12, 6.4), constrained_layout=True)
axs = axs.reshape(2, 3)

panels = [
    (f_true_hd * mask_hd, "Truth", "turbo", None),
    (f_lin_hd * mask_hd, f"linGP (SNR_rms={snr_rms_example:g})", "turbo", None),
    (f_log_hd * mask_hd, f"logGP (SNR_rms={snr_rms_example:g})", "turbo", None),
    (np.zeros_like(f_true_hd), "", "gray", None),
    (err_lin_hd * mask_hd, "linGP error", "RdBu_r", 0.4),
    (err_log_hd * mask_hd, "logGP error", "RdBu_r", 0.4),
]

for ax, (im, title, cmap, vmax) in zip(axs.flat, panels):
    if title == "":
        ax.axis("off")
        continue
    if vmax is None:
        ax.imshow(im, **im_kwargs, cmap=cmap)
    else:
        ax.imshow(im, **im_kwargs, cmap=cmap, vmin=-vmax, vmax=vmax)
    k.plt_rt1_flux(ax=ax, linewidths=0.6)
    ax.set_title(title)

plt.show()


In [ ]:
# Optional: line profile comparison at z ≈ 0
z0 = 0.0
iz = np.argmin(np.abs(problem["z_plot"] - z0))

fig, ax = plt.subplots(figsize=(6, 3.3), constrained_layout=True)
ax.plot(problem["r_plot"], f_true_hd[iz, :], lw=2, label="truth")
ax.plot(problem["r_plot"], f_lin_hd[iz, :], lw=1.5, label="linGP")
ax.plot(problem["r_plot"], f_log_hd[iz, :], lw=1.5, label="logGP")
ax.set_xlabel("R [m]")
ax.set_ylabel("emissivity / density (a.u.)")
ax.set_title(f"Midplane profile (z≈{problem['z_plot'][iz]:.3f}, SNR_rms={snr_rms_example:g})")
ax.grid(alpha=0.3)
ax.legend()
plt.show()


## メモ

- `logGP` は positivity を満たす一方で、Laplace 反復の計算コストがあります。
- `linGP` は閉形式で速いですが、低 SNR では負値が増えやすいです。
- 精密比較を行う場合は、`resolution`, `Lnum`, `n_trials`, `inducing_stride` を段階的に上げてください。
